In [ ]:
from scapy.all import sniff, IP, TCP
import threading
import queue
import config

class PacketCapture:
    def __init__(self):
        self.packet_queue = queue.Queue()
        self.stop_capture = threading.Event()

    def packet_callback(self, packet):
        if IP in packet and TCP in packet:
            self.packet_queue.put(packet)

    def start_capture(self, interface="wlp2s0"):
        def capture_thread():
            sniff(iface=interface,
                  prn=self.packet_callback,
                  store=0,
                  stop_filter=lambda _: self.stop_capture.is_set())

        self.capture_thread = threading.Thread(target=capture_thread)
        self.capture_thread.start()

    def stop(self):
        self.stop_capture.set()
        self.capture_thread.join()



In [14]:
packet_capture = PacketCapture()
packet_capture.start_capture()

packets_captured = []
for i in range(10):
    try:
        packet = packet_capture.packet_queue.get(timeout=5)
        packets_captured.append(packet)
        print(f"Paquet {i+1}: {packet.summary()}")
    except queue.Empty:
        print("Pas de paquet reçu")
        break

packet_capture.stop()

print(f"\nTotal de paquets capturés: {len(packets_captured)}")

Exception in thread Thread-29 (capture_thread):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/user/1000/ipykernel_164320/2368054729.py", line 17, in capture_thread
  File "/home/wmzy6987/Documents/Projects/IDS-Intrusion-Detection-System-/.venv/lib/python3.12/site-packages/scapy/sendrecv.py", line 1438, in sniff
    sniffer._run(*args, **kwargs)
  File "/home/wmzy6987/Documents/Projects/IDS-Intrusion-Detection-System-/.venv/lib/python3.12/site-packages/scapy/sendrecv.py", line 1283, in _run
    sniff_sockets[_RL2(iface)(type=ETH_P_ALL, iface=iface,
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/wmzy6987/Documents/Projects/IDS-Intrusion-Detection-System-/.venv/lib/python3.12/site-packages/scapy/arch/linux/__init__.py", line 219, in __init__
    self.ins = socket.socke

Pas de paquet reçu

Total de paquets capturés: 0
